In [51]:
function IR(A, b, x0; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    x = x0

    for i in 1:imax
        # Step 1: compute residual in ur precision
        r = ur.(b) - ur.(A) * ur.(x)

        # Step 2: solve d in ul precision
        d = ul.(A) \ ul.(r)
        # d = cu.(ul.(A)) \ cu.(ul.(r)) # windows 
        
        # Step 3: update x in u precision
        x = u.(x) + u.(d)
        
        if abs(norm(r)) < atol
            println("Converged at iter $(i)")
            break
        end
    end

    return x
end

IR (generic function with 1 method)

In [53]:
function example(n, m; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    A = rand(n, m)
    b = rand(n)
    x0 = ones(m)

    start_time = time()
    x_true = A \ b
    elapsed = time() - start_time
    x_true = vec(x_true)
    println("Run time: ", elapsed)

    start_time = time()
    x = IR(A, b, x0)
    elapsed = time() - start_time
    println("Run time: ", elapsed)
    
    error = abs(norm(x-x_true))
    println("Error: ", error)
    

end    

example (generic function with 1 method)

In [79]:
# import Pkg; Pkg.add("CUDA")
using LinearAlgebra
using BenchmarkTools
using CUDA

example(100, 100)

Run time: 0.00040411949157714844
Run time: 0.006175994873046875
Error: 1.3418050592868143e-5


In [258]:
n = 1000
m = 1000
A = rand(n, m)
B = rand(m, n)

A_32 = Float32.(A)
B_32 = Float32.(B)

A_16 = Float16.(A)
B_16 = Float16.(B)

start_time = time()
C = A * B
elapsed = time() - start_time
println(typeof(C))
println(C[1, 1])
println("Run time for precision ", typeof(A), ": ", elapsed, " seconds")

start_time = time()
C = A_32 * B_32
elapsed = time() - start_time
println(typeof(C))
println(C[1, 1])
println("Run time for precision ", typeof(A_32), ": ", elapsed, " seconds")

start_time = time()
C = A_16 * B_16
# C = cu.(A_16) * cu.(B_16) # windows 
elapsed = time() - start_time
println(typeof(C))
println(C[1, 1])
println("Run time for precision ", typeof(A_16), ": ", elapsed, " seconds")


Matrix{Float64}
241.05746925397526
Run time for precision Matrix{Float64}: 0.031625986099243164 seconds
Matrix{Float32}
241.05742
Run time for precision Matrix{Float32}: 0.009810924530029297 seconds
Matrix{Float16}
239.0
Run time for precision Matrix{Float16}: 0.04334115982055664 seconds
